# TFM — Notebook 03: modelos de aprendizaje automático

**Predicción de demanda energética mediante Machine Learning y datos abiertos**

Segunda fase del modelado. Se evalúan modelos de aprendizaje automático sobre
las variables construidas en el notebook 01, empleando **el mismo esquema de
validación y las mismas métricas** que el notebook 02, de modo que la
comparación sea directa.

---

### Resultados de la fase anterior

| Modelo | MAPE mediana | Frente al ingenuo |
|---|---|---|
| SARIMA clásico | 2,42 % | +20,9 % |
| Ingenuo semanal | 3,06 % | referencia |
| Ingenuo diario | 3,21 % | −5,0 % |
| Prophet | 4,78 % | −56,4 % |
| SARIMAX + Fourier | 4,90 % | −60,3 % |

El análisis por ventana reveló dos carencias que estos modelos no resuelven:

- El ingenuo diario **falla los lunes** (errores del 15,4 % y 13,9 %), porque
  los predice a partir del domingo.
- SARIMAX **falla los domingos** (11,2 % y 13,0 %): con dos armónicos no puede
  representar el escalón entre laborables y fin de semana.

Ambas son limitaciones de calendario. Los modelos de esta fase disponen de
`festivo`, `laborable` y `vispera_festivo` como variables, por lo que deberían
corregirlas.

---
## 0. Configuración

In [ ]:
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (13, 4.5)
plt.rcParams["figure.dpi"]     = 110
plt.rcParams["axes.grid"]      = True
plt.rcParams["grid.alpha"]     = 0.3
plt.rcParams["font.size"]      = 10
sns.set_palette("deep")

print("Librerías cargadas.")

In [ ]:
RAIZ = Path.cwd()
if RAIZ.name == "notebooks":
    RAIZ = RAIZ.parent

DIR_DATOS      = RAIZ / "datos"
DIR_FIGURAS    = RAIZ / "figuras";    DIR_FIGURAS.mkdir(exist_ok=True)
DIR_RESULTADOS = RAIZ / "resultados"; DIR_RESULTADOS.mkdir(exist_ok=True)

def guardar_figura(nombre):
    plt.savefig(DIR_FIGURAS / f"{nombre}.png", dpi=200, bbox_inches="tight")
    print(f"  guardada -> {nombre}.png")

# Mismos parámetros que el notebook 02: la comparación debe ser homogénea
HORIZONTE  = 24
N_VENTANAS = 14
PASO       = 120

assert PASO % 168 != 0, "PASO no puede ser múltiplo de una semana"
print(f"Horizonte {HORIZONTE} h | {N_VENTANAS} orígenes | paso {PASO//24} días")

---
## 1. Carga de datos

In [ ]:
RUTA = DIR_DATOS / "dataset_modelado.parquet"
if not RUTA.exists():
    raise FileNotFoundError(f"Falta {RUTA}. Ejecuta antes el notebook 01.")

df = pd.read_parquet(RUTA)
if df.index.tz is not None:
    df.index = df.index.tz_localize(None)

y = df["demanda_mw"]

print(f"Registros: {len(df):,}   Columnas: {df.shape[1]}")
print(f"Periodo  : {y.index.min():%Y-%m-%d} a {y.index.max():%Y-%m-%d}")
print(f"\nColumnas disponibles:")
print(sorted(df.columns.tolist()))

In [ ]:
# ── Métricas: idénticas a las del notebook 02 ────────────────────────────
def metricas(y_real, y_pred):
    y_real = np.asarray(y_real, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    e = y_real - y_pred
    return {
        "MAE":   float(np.mean(np.abs(e))),
        "RMSE":  float(np.sqrt(np.mean(e ** 2))),
        "MAPE":  float(np.mean(np.abs(e / y_real)) * 100),
        "sMAPE": float(np.mean(2*np.abs(e)/(np.abs(y_real)+np.abs(y_pred))) * 100),
    }


def generar_ventanas(n_total, horizonte, n_ventanas, paso):
    v = []
    for i in range(n_ventanas):
        a = n_total - (n_ventanas - i) * paso
        b = min(a + horizonte, n_total)
        if a > 0 and b > a:
            v.append((a, b))
    return v


VENTANAS = generar_ventanas(len(y), HORIZONTE, N_VENTANAS, PASO)

dias = pd.Series([y.index[a].day_name() for a, _ in VENTANAS]).value_counts()
print(f"Orígenes: {len(VENTANAS)} | días de la semana cubiertos: {len(dias)} de 7")
assert len(dias) == 7, "La evaluación no cubre toda la semana"
print(f"Periodo evaluado: {y.index[VENTANAS[0][0]]:%d-%b-%Y} "
      f"a {y.index[VENTANAS[-1][1]-1]:%d-%b-%Y}")

---
## 2. Selección de variables: la fuga de información

Este es **el punto más delicado de todo el trabajo**, y el error más frecuente
en la aplicación de aprendizaje automático a series temporales.

### El problema

El notebook 01 construyó desfases de 1, 2, 3, 24, 25, 48 y 168 horas. Al
predecir con 24 horas de antelación, el instante $t+24$ debe estimarse desde
el instante $t$. En ese momento:

| Variable en $t+24$ | Equivale a | ¿Se conoce en $t$? |
|---|---|---|
| `lag_1h` | $y_{t+23}$ | **No** |
| `lag_2h` | $y_{t+22}$ | **No** |
| `lag_24h` | $y_{t}$ | Sí |
| `lag_168h` | $y_{t-144}$ | Sí |
| `media_24h` | media de $t+1 \ldots t+23$ | **No** |
| `media_168h` | ventana que llega hasta $t+23$ | **No** |
| `hora`, `festivo`, `tmed` | calendario y meteorología | Sí |

Utilizar las variables de la columna «No» produce métricas excelentes e
irreproducibles en producción: el modelo estaría usando el futuro para predecir
el futuro.

### La regla

**Para un horizonte de $h$ horas, solo son válidos los desfases de $h$ o más, y
los estadísticos móviles calculados sobre ventanas que terminen antes de $t$.**

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 2.1 — Clasificación de las variables disponibles
# ═══════════════════════════════════════════════════════════════════════════
import re

def clasificar_variables(columnas, horizonte):
    """Separa las variables utilizables de las que provocarían fuga."""
    seguras, con_fuga = [], []

    for c in columnas:
        if c == "demanda_mw":
            continue

        # Desfases: válidos solo si el retardo es >= horizonte
        m = re.fullmatch(r"lag_(\d+)h", c)
        if m:
            (seguras if int(m.group(1)) >= horizonte else con_fuga).append(c)
            continue

        # Estadísticos móviles del notebook 01: se calcularon con shift(1),
        # de modo que su ventana termina en t+h-1 e incluye horas aún
        # desconocidas. Se descartan todos y se recalculan en la celda 2.2
        # con el desplazamiento correcto.
        if re.fullmatch(r"(media|std)_(\d+)h", c):
            con_fuga.append(c)
            continue

        # Variables auxiliares del EDA que no deben entrar al modelo
        if c in ("esperado", "residuo", "z_robusto", "atipico",
                 "tipo_anomalia", "fecha"):
            continue

        # Calendario y meteorología: conocidas de antemano
        seguras.append(c)

    return sorted(seguras), sorted(con_fuga)


SEGURAS, CON_FUGA = clasificar_variables(df.columns, HORIZONTE)

# Solo columnas numéricas
SEGURAS = [c for c in SEGURAS if pd.api.types.is_numeric_dtype(df[c])]

print(f"Variables utilizables ({len(SEGURAS)}):")
for c in SEGURAS:
    print(f"   {c}")

print(f"\nDescartadas por fuga de información ({len(CON_FUGA)}):")
for c in CON_FUGA:
    print(f"   {c}")

### 2.2 Recálculo de los estadísticos móviles

Los estadísticos móviles del notebook 01 se construyeron con un desplazamiento
de una hora, adecuado para predicción a un paso pero **no para 24 horas**: la
ventana de `media_24h` en $t+24$ abarca de $t$ a $t+23$, tramo en su mayor parte
desconocido en el momento de predecir.

Se recalculan aquí desplazando la serie **el horizonte completo**, de modo que
toda la ventana quede cerrada antes del origen de predicción.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Estadísticos móviles con desplazamiento igual al horizonte
# ═══════════════════════════════════════════════════════════════════════════
base = y.shift(HORIZONTE)      # solo información anterior al origen

for ventana in [24, 168]:
    df[f"media{ventana}_h{HORIZONTE}"] = base.rolling(ventana).mean()
    df[f"std{ventana}_h{HORIZONTE}"]   = base.rolling(ventana).std()

NUEVAS = [f"media24_h{HORIZONTE}", f"std24_h{HORIZONTE}",
          f"media168_h{HORIZONTE}", f"std168_h{HORIZONTE}"]
SEGURAS = sorted(set(SEGURAS) | set(NUEVAS))

print("Variables recalculadas:")
for c in NUEVAS:
    print(f"   {c:22} nulos: {df[c].isna().sum():>5}")

print(f"\nTotal de variables utilizables: {len(SEGURAS)}")

# Verificación: ninguna variable puede correlacionar perfectamente con y
corr = df[SEGURAS].corrwith(y).abs().sort_values(ascending=False)
print(f"\nCorrelación más alta con la variable objetivo: "
      f"{corr.iloc[0]:.3f} ({corr.index[0]})")
if corr.iloc[0] > 0.995:
    print("ATENCIÓN: correlación casi perfecta. Revisa si hay fuga.")

### 2.3 Demostración empírica

Conviene cuantificar el efecto en lugar de asumirlo. Se entrena el mismo modelo
con ambos conjuntos de variables y se comparan los resultados.

In [ ]:
def evaluar_rapido(columnas, n_ventanas=6):
    """Evaluación abreviada para ilustrar el efecto de la fuga."""
    apes = []
    for a, b in VENTANAS[-n_ventanas:]:
        Xtr, ytr = df[columnas].iloc[:a], y.iloc[:a]
        Xte, yte = df[columnas].iloc[a:b], y.iloc[a:b]
        valido = Xtr.notna().all(axis=1)
        m = LGBMRegressor(n_estimators=300, verbose=-1, random_state=42)
        m.fit(Xtr[valido], ytr[valido])
        p = m.predict(Xte)
        apes.append(np.mean(np.abs((yte.values - p) / yte.values)) * 100)
    return float(np.mean(apes))


if CON_FUGA:
    mape_fuga  = evaluar_rapido(SEGURAS + CON_FUGA)
    mape_limpio = evaluar_rapido(SEGURAS)

    print(f"{'Conjunto de variables':45} {'MAPE':>8}")
    print("-" * 55)
    print(f"{'Incluyendo desfases cortos (FUGA)':45} {mape_fuga:7.2f} %")
    print(f"{'Solo variables conocidas en el origen':45} {mape_limpio:7.2f} %")
    print("-" * 55)

    inflacion = (mape_limpio - mape_fuga) / mape_limpio * 100
    print(f"\nLa fuga haría parecer al modelo un {inflacion:.0f} % mejor de lo que es.")
    print("En despliegue real solo se alcanzaría el segundo valor.")
else:
    print("No hay variables con riesgo de fuga en este dataset.")

> **Para la memoria.** Este experimento merece un apartado propio. Documenta
> una decisión metodológica que la mayoría de trabajos ni siquiera menciona, y
> cuantifica las consecuencias de no tomarla. Es el tipo de rigor que distingue
> un trabajo sólido.

---
## 3. Modelos

Se sigue la progresión de complejidad indicada por la dirección del TFT:

| Modelo | Familia | Aportación |
|---|---|---|
| **Ridge** | Lineal regularizado | Referencia lineal. Solo captura relaciones aditivas |
| **Random Forest** | Conjunto de árboles | Captura no linealidades e interacciones |
| **XGBoost** | Potenciación del gradiente | Ajuste secuencial de los errores |
| **LightGBM** | Potenciación del gradiente | Variante más rápida, habitual en competiciones |

Se emplean configuraciones conservadoras y comunes. La optimización de
hiperparámetros se aborda en la sección 6, una vez identificado el modelo más
prometedor.

In [ ]:
SEMILLA = 42

MODELOS = {
    "Ridge": lambda: make_pipeline(
        StandardScaler(),
        Ridge(alpha=1.0, random_state=SEMILLA)),

    # Configuración contenida: Random Forest reentrena 14 veces sobre 26.000
    # filas. Con 300 árboles de profundidad 18 el ajuste supera los tres
    # minutos por ventana, lo que haría el notebook impracticable.
    "Random Forest": lambda: RandomForestRegressor(
        n_estimators=100, max_depth=12, min_samples_leaf=10,
        max_features=0.5, n_jobs=-1, random_state=SEMILLA),

    "XGBoost": lambda: XGBRegressor(
        n_estimators=400, learning_rate=0.05, max_depth=7,
        subsample=0.8, colsample_bytree=0.8, tree_method="hist",
        n_jobs=-1, random_state=SEMILLA, verbosity=0),

    "LightGBM": lambda: LGBMRegressor(
        n_estimators=400, learning_rate=0.05, num_leaves=48,
        subsample=0.8, colsample_bytree=0.8,
        n_jobs=-1, random_state=SEMILLA, verbose=-1),
}

print(f"{len(MODELOS)} modelos definidos sobre {len(SEGURAS)} variables.\n")
print("Tiempo orientativo de la sección 4 (14 reentrenamientos por modelo):")
print("   Ridge          <  1 min")
print("   Random Forest  ~ 10 min   <- el más lento")
print("   XGBoost        ~  2 min")
print("   LightGBM       ~  1 min")

---
## 4. Evaluación con origen rodante

Mismo procedimiento que el notebook 02. En cada origen se reentrena el modelo
con todo el histórico disponible y se predicen las 24 horas siguientes.

Se registran **las métricas dentro y fuera de muestra**, tal como indicó la
dirección del TFT. A diferencia de los modelos estadísticos, aquí ambas se
calculan sobre el mismo horizonte, por lo que la comparación es informativa
respecto al sobreajuste.

In [ ]:
def backtesting(nombre, constructor, columnas, verbose=True):
    """Origen rodante registrando métricas dentro y fuera de muestra."""
    filas_out, filas_in, predicciones = [], [], []
    t0 = time.time()

    for i, (a, b) in enumerate(VENTANAS):
        Xtr, ytr = df[columnas].iloc[:a], y.iloc[:a]
        Xte, yte = df[columnas].iloc[a:b], y.iloc[a:b]

        valido = Xtr.notna().all(axis=1) & ytr.notna()
        Xtr, ytr = Xtr[valido], ytr[valido]

        modelo = constructor()
        modelo.fit(Xtr, ytr)

        pred_out = modelo.predict(Xte)
        pred_in  = modelo.predict(Xtr)

        filas_out.append(metricas(yte.values, pred_out))
        filas_in.append(metricas(ytr.values, pred_in))

        predicciones.append(pd.DataFrame(
            {"real": yte.values, "pred": pred_out, "ventana": i},
            index=yte.index))

    return {
        "Modelo":   nombre,
        "fuera":    pd.DataFrame(filas_out),
        "dentro":   pd.DataFrame(filas_in),
        "pred":     pd.concat(predicciones),
        "segundos": time.time() - t0,
    }


resultados = {}
for nombre, constructor in MODELOS.items():
    print(f"Evaluando {nombre}...", end=" ", flush=True)
    r = backtesting(nombre, constructor, SEGURAS)
    resultados[nombre] = r
    print(f"MAPE = {r['fuera']['MAPE'].mean():.2f} %   ({r['segundos']:.0f} s)")

print("\nEvaluación completada.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 4.1 — Tabla de resultados
# ═══════════════════════════════════════════════════════════════════════════
filas = []
for nombre, r in resultados.items():
    p = r["pred"].copy()
    p["ape"] = np.abs((p["real"] - p["pred"]) / p["real"]) * 100
    por_ventana = p.groupby("ventana")["ape"].mean()

    filas.append({
        "Modelo":        nombre,
        "MAPE medio":    por_ventana.mean(),
        "MAPE mediana":  por_ventana.median(),
        "MAPE std":      por_ventana.std(),
        "MAPE peor":     por_ventana.max(),
        "MAE":           r["fuera"]["MAE"].mean(),
        "RMSE":          r["fuera"]["RMSE"].mean(),
        "MAPE train":    r["dentro"]["MAPE"].mean(),
        "Segundos":      r["segundos"],
    })

tabla_ml = pd.DataFrame(filas).set_index("Modelo").sort_values("MAPE mediana")
display(tabla_ml.round(2))

---
## 5. Comparación con la línea base

Se recuperan los resultados del notebook 02 para construir la tabla conjunta.

In [ ]:
ruta_base = DIR_RESULTADOS / "comparativa_baseline.csv"

if ruta_base.exists():
    tabla_base = pd.read_csv(ruta_base, index_col=0)
    columnas = ["MAPE medio", "MAPE mediana", "MAPE std", "MAPE peor", "MAE", "RMSE"]

    conjunta = pd.concat([
        tabla_base[columnas].assign(Fase="Línea base"),
        tabla_ml[columnas].assign(Fase="Machine learning"),
    ]).sort_values("MAPE mediana")

    referencia = tabla_base.loc[
        [i for i in tabla_base.index if i.startswith(("Ingenuo", "Perfil"))],
        "MAPE mediana"].min()
    conjunta["vs ingenuo"] = (referencia - conjunta["MAPE mediana"]) / referencia * 100

    print(f"Referencia (mejor ingenuo): {referencia:.2f} %\n")
    display(conjunta.round(2))

    conjunta.to_csv(DIR_RESULTADOS / "comparativa_completa.csv")
else:
    print("No se encuentra comparativa_baseline.csv. Ejecuta antes el notebook 02.")
    conjunta = tabla_ml.copy()
    referencia = tabla_ml["MAPE mediana"].max()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colores = {"Línea base": "#888780", "Machine learning": "#4C72B0"}
c = [colores.get(f, "#4C72B0") for f in conjunta.get("Fase", ["Machine learning"]*len(conjunta))]

axes[0].barh(conjunta.index, conjunta["MAPE mediana"], color=c,
             xerr=conjunta["MAPE std"], error_kw={"lw": 1, "alpha": 0.45})
axes[0].axvline(referencia, color="crimson", ls="--", lw=1.2,
                label=f"Mejor ingenuo ({referencia:.2f} %)")
axes[0].set_xlabel("MAPE mediana (%)")
axes[0].set_title("Comparativa global", fontweight="bold")
axes[0].legend(fontsize=8)
axes[0].invert_yaxis()

# Entrenamiento frente a producción (solo ML)
x = np.arange(len(tabla_ml))
ancho = 0.38
axes[1].bar(x - ancho/2, tabla_ml["MAPE train"], ancho,
            label="Entrenamiento", color="#55A868")
axes[1].bar(x + ancho/2, tabla_ml["MAPE medio"], ancho,
            label="Producción", color="#C44E52")
axes[1].set_xticks(x)
axes[1].set_xticklabels(tabla_ml.index, rotation=15, fontsize=9)
axes[1].set_ylabel("MAPE (%)")
axes[1].set_title("Entrenamiento frente a producción", fontweight="bold")
axes[1].legend(fontsize=8)

plt.tight_layout()
guardar_figura("14_comparativa_ml")
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 5.1 — Control de sobreajuste  [TUTOR]
# ═══════════════════════════════════════════════════════════════════════════
print("Diferencia entre error de entrenamiento y de producción\n")

for nombre in tabla_ml.index:
    e = tabla_ml.loc[nombre, "MAPE train"]
    p = tabla_ml.loc[nombre, "MAPE medio"]
    rel = (p - e) / e * 100 if e else np.nan

    if rel < 50:
        veredicto = "sin indicios de sobreajuste"
    elif rel < 150:
        veredicto = "diferencia moderada"
    else:
        veredicto = "SOBREAJUSTE: revisar regularización"

    print(f"  {nombre:16} {e:5.2f} % -> {p:5.2f} %   ({rel:+6.0f} %)  {veredicto}")

print("\nUn error de entrenamiento muy bajo frente al de producción indica que")
print("el modelo memoriza en lugar de generalizar. En modelos de árboles se")
print("corrige limitando la profundidad o aumentando min_samples_leaf.")

---
## 6. Importancia de las variables

Se analiza el modelo con mejor rendimiento para identificar qué factores
determinan la predicción. Este apartado alimenta directamente el objetivo
específico de *identificar los factores que más influyen en la demanda*.

In [ ]:
MEJOR = tabla_ml.index[0]
print(f"Modelo seleccionado: {MEJOR}\n")

# Entrenamiento sobre todo el histórico previo al último origen
corte = VENTANAS[-1][0]
Xtr, ytr = df[SEGURAS].iloc[:corte], y.iloc[:corte]
valido = Xtr.notna().all(axis=1) & ytr.notna()
Xtr, ytr = Xtr[valido], ytr[valido]

modelo_final = MODELOS[MEJOR]()
modelo_final.fit(Xtr, ytr)

# Importancia nativa (no disponible en modelos lineales)
estimador = modelo_final[-1] if hasattr(modelo_final, "steps") else modelo_final

if hasattr(estimador, "feature_importances_"):
    imp = pd.Series(estimador.feature_importances_,
                    index=SEGURAS).sort_values(ascending=False)

    plt.figure(figsize=(10, max(4, len(imp) * 0.28)))
    imp.head(20).sort_values().plot(kind="barh", color="#4C72B0")
    plt.xlabel("Importancia")
    plt.title(f"Importancia de variables — {MEJOR}", fontweight="bold")
    plt.tight_layout()
    guardar_figura("15_importancia_variables")
    plt.show()

    print("Diez variables más influyentes:\n")
    print(imp.head(10).round(4).to_string())
else:
    print(f"{MEJOR} no expone importancia de variables.")

### 6.1 Valores SHAP

La importancia nativa indica **cuánto** se usa cada variable, pero no en qué
dirección ni en qué condiciones. Los valores SHAP \parencite{lundberg2017shap}
descomponen cada predicción individual en la contribución de cada variable.

In [ ]:
try:
    import shap

    # Muestra representativa: el cálculo exacto sobre 26.000 filas es lento
    muestra = Xtr.sample(min(2000, len(Xtr)), random_state=SEMILLA)

    if MEJOR in ("XGBoost", "LightGBM", "Random Forest"):
        explicador = shap.TreeExplainer(estimador)
        valores = explicador.shap_values(muestra)

        shap.summary_plot(valores, muestra, show=False, max_display=15)
        plt.title(f"Valores SHAP — {MEJOR}", fontweight="bold")
        plt.tight_layout()
        guardar_figura("16_shap_summary")
        plt.show()

        print("\nCada punto es una observación. El color indica el valor de la")
        print("variable (rojo alto, azul bajo) y la posición horizontal su efecto")
        print("sobre la predicción.")
    else:
        print(f"SHAP con TreeExplainer no aplica a {MEJOR}.")

except ImportError:
    print("Instala shap:  %pip install shap")
except Exception as e:
    print(f"No se pudo calcular SHAP: {type(e).__name__}: {e}")

---
## 7. Análisis de errores

El objetivo es comprobar si los modelos de aprendizaje automático corrigen las
carencias detectadas en la fase anterior: los fallos del ingenuo diario en lunes
y los de SARIMAX en domingo.

In [ ]:
# Contexto de calendario de cada ventana
contexto = []
for i, (a, b) in enumerate(VENTANAS):
    tramo = df.iloc[a:b]
    fecha = y.index[a]
    contexto.append({
        "ventana": i,
        "fecha":   fecha.date(),
        "dia":     fecha.day_name(),
        "festivo": int(tramo["festivo"].max()) if "festivo" in df.columns else 0,
        "finde":   int(fecha.dayofweek >= 5),
    })

CONTEXTO = pd.DataFrame(contexto).set_index("ventana")
CONTEXTO["tipo"] = np.select(
    [CONTEXTO["festivo"] == 1, CONTEXTO["finde"] == 1],
    ["Festivo", "Fin de semana"], default="Laborable")

# Error por tipo de día
desglose = []
for nombre, r in resultados.items():
    p = r["pred"].copy()
    p["ape"] = np.abs((p["real"] - p["pred"]) / p["real"]) * 100
    por_v = p.groupby("ventana")["ape"].mean().rename("ape")
    unido = pd.concat([por_v, CONTEXTO["tipo"]], axis=1).dropna()

    fila = {"Modelo": nombre}
    for tipo in ["Laborable", "Fin de semana", "Festivo"]:
        sel = unido[unido["tipo"] == tipo]["ape"]
        fila[tipo] = sel.mean() if len(sel) else np.nan
    desglose.append(fila)

por_tipo = pd.DataFrame(desglose).set_index("Modelo").loc[tabla_ml.index]
print("MAPE según el tipo de día\n")
display(por_tipo.round(2))

# Error por día de la semana del mejor modelo
p = resultados[MEJOR]["pred"].copy()
p["ape"] = np.abs((p["real"] - p["pred"]) / p["real"]) * 100
por_v = p.groupby("ventana")["ape"].mean().rename("ape")
detalle = pd.concat([por_v, CONTEXTO[["dia", "fecha"]]], axis=1).dropna()

print(f"\nError por origen — {MEJOR}\n")
print(detalle.sort_values("ape", ascending=False).round(2).to_string())

In [ ]:
# ── Predicción frente a valor real ────────────────────────────────────────
comparar = [m for m in [MEJOR, "Ridge"] if m in resultados][:2]

n = min(4, len(VENTANAS))
fig, axes = plt.subplots(n, 1, figsize=(13, 2.6 * n))
if n == 1:
    axes = [axes]

for k in range(n):
    ax = axes[k]
    iv = len(VENTANAS) - n + k
    base = resultados[comparar[0]]["pred"]
    tramo = base[base["ventana"] == iv]

    ax.plot(tramo.index, tramo["real"], color="black", lw=2,
            marker="o", ms=3, label="Real")
    for m in comparar:
        q = resultados[m]["pred"]
        q = q[q["ventana"] == iv]
        ax.plot(q.index, q["pred"], lw=1.5, ls="--", label=m)

    dia = CONTEXTO.loc[iv, "dia"]
    ax.set_title(f"{tramo.index[0]:%d-%b-%Y} ({dia})", fontweight="bold", fontsize=10)
    ax.set_ylabel("MW")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    if k == 0:
        ax.legend(fontsize=8, ncol=3)

plt.tight_layout()
guardar_figura("17_predicciones_ml")
plt.show()

In [ ]:
# ── Error por hora del día ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for nombre in tabla_ml.index:
    q = resultados[nombre]["pred"].copy()
    q["hora"] = q.index.hour
    q["ape"] = np.abs((q["real"] - q["pred"]) / q["real"]) * 100
    perfil = q.groupby("hora")["ape"].mean()
    axes[0].plot(perfil.index, perfil.values, marker="o", ms=3, lw=1.4, label=nombre)

axes[0].set_xlabel("Hora del día")
axes[0].set_ylabel("Error absoluto porcentual (%)")
axes[0].set_title("Error medio por hora", fontweight="bold")
axes[0].legend(fontsize=8)

por_tipo.plot(kind="barh", ax=axes[1], width=0.75,
              color=["#55A868", "#DD8452", "#C44E52"])
axes[1].set_xlabel("MAPE (%)")
axes[1].set_title("Error según el tipo de día", fontweight="bold")
axes[1].legend(fontsize=8)
axes[1].invert_yaxis()

plt.tight_layout()
guardar_figura("18_error_ml")
plt.show()

---
## 8. Guardado de resultados

In [ ]:
todas = pd.concat(
    [r["pred"].assign(modelo=m) for m, r in resultados.items()]
).reset_index().rename(columns={"index": "datetime"})

todas.to_parquet(DIR_RESULTADOS / "predicciones_ml.parquet", index=False)
tabla_ml.to_csv(DIR_RESULTADOS / "comparativa_ml.csv")
por_tipo.to_csv(DIR_RESULTADOS / "error_por_tipo_dia.csv")

print(f"Predicciones guardadas: {len(todas):,} filas\n")
print("Archivos generados:")
for f in ["predicciones_ml.parquet", "comparativa_ml.csv",
          "error_por_tipo_dia.csv", "comparativa_completa.csv"]:
    ruta = DIR_RESULTADOS / f
    if ruta.exists():
        print(f"   {f}")

print(f"\nResumen\n")
print(tabla_ml[["MAPE mediana", "MAPE std", "MAE"]].round(2).to_string())

---
## Conclusiones y siguientes pasos

### Qué reportar en la memoria

1. **La demostración de fuga de información** (sección 2.2). Es un apartado con
   valor propio: cuantifica una decisión metodológica que raramente se explicita.

2. **La comparación con la línea base** (sección 5). El porcentaje de mejora
   frente al mejor modelo ingenuo es la cifra que resume el valor del trabajo.

3. **El control de sobreajuste** (sección 5.1). Aquí la comparación entre error
   de entrenamiento y de producción sí es informativa, porque ambos se calculan
   sobre el mismo horizonte.

4. **La importancia de variables y los valores SHAP** (sección 6), que responden
   al objetivo específico de identificar los factores determinantes.

5. **El error por tipo de día** (sección 7): comprobar si se corrigen los fallos
   en lunes y domingo detectados en la fase anterior.

### Posibles mejoras

Si el rendimiento no alcanza el objetivo del 8 % de MAPE fijado en los
objetivos del trabajo:

- **Optimización de hiperparámetros** con Optuna sobre el modelo más prometedor.
- **Combinación de modelos**: promediar las predicciones de los mejores suele
  mejorar la robustez, hallazgo recurrente en las competiciones M.
- **Predicción del incremento** en lugar del valor absoluto, reformulación que
  la literatura reciente señala como beneficiosa en horizontes cortos.

### Notebook 04

Aplicación web en Streamlit con el modelo seleccionado.